In [1]:
from pytorch_lightning.trainer.trainer import Trainer
from pytorch_lightning.callbacks import EarlyStopping
import optuna

%load_ext autoreload
%autoreload 2

from Model import Model
from Datamodule import Datamodule

/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUT_FEATURES = 10

In [3]:
datamodule = Datamodule()
datamodule.prepare_data()
datamodule.setup("fit")

In [4]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    score = 0.0
    for fold, train_loader, val_loader in datamodule.cv_train_val_splits():
        model = Model(1, 2, OUT_FEATURES, lr)

        trainer = Trainer(
            max_epochs=10,
            fast_dev_run=False,
        )
        trainer.fit(model, train_loader, val_loader)
        result = trainer.validate(model, val_loader)

        assert len(result) == 1
        accuracy = result[0]["val_accuracy"]
        score += accuracy

    avg_score = score / 5
    return avg_score

In [5]:
study = optuna.create_study(study_name="hparams", direction="maximize")
study.optimize(objective, n_trials=3)

[I 2026-06-22 20:09:10,976] A new study created in memory with name: hparams
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('AMD Radeon RX 7700 XT') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name     | Type               | Params
-----------------------------------------

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 188/188 [00:00<00:00, 202.49it/s, v_num=14]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 188/188 [00:00<00:00, 224.06it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 188/188 [00:00<00:00, 228.16it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 188/188 [00:00<00:00, 214.38it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 188/188 [00:00<00:00, 189.78it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 188/188 [00:00<00:00, 228.39it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 6: 100%|██████████| 188/188 [00:00<00:00, 234.82it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 7: 100%|██████████| 188/188 [00:00<00:00, 226.57it/s, v_num=14]    
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 8: 100%|██████████| 188/188 [00:00<00:

...

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 188/188 [00:00<00:00, 219.03it/s, v_num=28]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Validation DataLoader 0: 100%|██████████| 24/24 [00:00<00:00, 353.42it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      val_accuracy          0.10000000149011612
        val_loss            2.3085031509399414
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[I 2026-06-22 20:11:33,618] Trial 2 finished with value: 0.16255000233650208 and parameters: {'lr': 0.08051584960463676}. Best is trial 0 with value: 0.4420833349227905.


In [7]:
study.best_value, study.best_params

(0.4420833349227905, {'lr': 0.00034233283131945787})